# Raw Data Exploration and Profiling

## 1. Purpose and Scope

This notebook explores the raw eBay Browse API data landed by DLTHub in Google Cloud Storage. The goal is to understand source structure, volume, completeness, identifier behavior, and relationships between discovery results and enriched item details before designing Bronze and Silver transformations.

**Scope**
- Profile the `browse_search` and `item_details` raw resources.
- Preserve the source representation and DLTHub technical metadata during exploration.
- Record observed behavior and limitations to inform downstream data modeling.

## 2. Data Loading

The following cells define the raw GCS location and load the two business-facing resources into Spark DataFrames. The raw files are read in place; this notebook does not alter them.


In [0]:
RAW_BUCKET = "gs://market-intelligence-raw"
print(RAW_BUCKET)

In [0]:
item_details_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/item_details/*.jsonl.gz"
)

display(item_details_df.limit(5))

In [0]:
browse_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/browse_search/*.jsonl.gz"
)

display(browse_df.limit(5))

## 3. Profiling Setup and Dataset Inventory

Define reusable references for the loaded DataFrames and establish baseline row and column counts. These measurements describe the data available at the time this notebook is run.


In [0]:
from pyspark.sql import functions as F

# ============================================================
# Dataset references
# ============================================================

DATASETS = {
    "browse_search": browse_df,
    "item_details": item_details_df,
}

for name, df in DATASETS.items():
    print(f"{name}:")
    print(f"  rows    : {df.count():,}")
    print(f"  columns : {len(df.columns):,}")
    print()

## 4. Schema Profiling

Inspect column names, Spark data types, and nullability for each source resource. This is a structural inventory, not a business-level type-casting or schema standardization step.


In [0]:
def profile_schema(df, dataset_name: str):
    rows = []

    for field in df.schema.fields:
        rows.append(
            (
                field.name,
                field.dataType.simpleString(),
                field.nullable,
            )
        )

    return spark.createDataFrame(
        rows,
        ["column_name", "data_type", "nullable"]
    ).withColumn(
        "dataset",
        F.lit(dataset_name)
    ).select(
        "dataset",
        "column_name",
        "data_type",
        "nullable",
    )

In [0]:
browse_schema = profile_schema(
    browse_df,
    "browse_search"
)

display(browse_schema)

In [0]:
item_details_schema = profile_schema(
    item_details_df,
    "item_details"
)

display(item_details_schema)

## 5. Completeness Profiling

Measure non-null and null values by column for both datasets. Nulls are treated as profiling signals; their meaning may depend on listing type, category, or API response behavior and should not automatically be classified as defects.


In [0]:
def profile_completeness(df):
    total_rows = df.count()

    expressions = []

    for column_name in df.columns:
        expressions.extend([
            F.count(F.col(column_name)).alias(f"{column_name}__non_null"),
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{column_name}__null"),
        ])

    result = df.agg(*expressions)

    profile_rows = []

    for column_name in df.columns:
        non_null_col = f"{column_name}__non_null"
        null_col = f"{column_name}__null"

        row = result.select(
            F.lit(column_name).alias("column_name"),
            F.col(non_null_col).alias("non_null_count"),
            F.col(null_col).alias("null_count"),
        ).withColumn(
            "total_rows",
            F.lit(total_rows)
        ).withColumn(
            "null_percentage",
            F.round(
                F.col("null_count") / F.col("total_rows") * 100,
                2
            )
        )

        profile_rows.append(row)

    final_df = profile_rows[0]

    for row in profile_rows[1:]:
        final_df = final_df.unionByName(row)

    return final_df.orderBy(
        F.col("null_percentage").desc()
    )

In [0]:
item_details_completeness = profile_completeness(item_details_df)

display(item_details_completeness)

## 6. Cardinality, Repetition, and Resource Coverage

Assess `item_id` cardinality, identify repeated IDs, inspect their frequency, and compare the distinct listing IDs present in Browse discovery versus item details.


In [0]:
from pyspark.sql import functions as F

# ============================================================
# Item ID cardinality
# ============================================================

for name, df in DATASETS.items():

    total_rows = df.count()

    distinct_items = (
        df.select("item_id")
        .where(F.col("item_id").isNotNull())
        .distinct()
        .count()
    )

    duplicate_rows = total_rows - distinct_items

    print(f"\n{name}")
    print("-" * 50)
    print(f"Total rows           : {total_rows:,}")
    print(f"Distinct item_ids    : {distinct_items:,}")
    print(f"Duplicate occurrences: {duplicate_rows:,}")

In [0]:
browse_item_frequency = (
    browse_df
    .groupBy("item_id")
    .count()
    .orderBy(F.col("count").desc())
)

display(browse_item_frequency.limit(20))

In [0]:
item_details_item_frequency = (
    item_details_df
    .groupBy("item_id")
    .count()
    .orderBy(F.col("count").desc())
)

display(item_details_item_frequency.limit(20))

In [0]:
repeated_items = (
    browse_df
    .groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

browse_duplicates = (
    browse_df
    .join(repeated_items, on="item_id", how="inner")
    .orderBy("item_id")
)

display(
    browse_duplicates.select(
        "item_id",
        "title",
        "price__value",
        "price__currency",
        "seller__username",
        "condition",
        "_dlt_load_id"
    ).limit(100)
)

In [0]:
browse_unique_items = (
    browse_df
    .select("item_id")
    .where(F.col("item_id").isNotNull())
    .distinct()
    .withColumn("in_browse", F.lit(True))
)

details_unique_items = (
    item_details_df
    .select("item_id")
    .where(F.col("item_id").isNotNull())
    .distinct()
    .withColumn("in_details", F.lit(True))
)

coverage = (
    browse_unique_items
    .join(details_unique_items, on="item_id", how="left")
    .fillna({"in_details": False})
)

display(
    coverage.groupBy("in_details")
    .count()
)

## 7. Repeated Browse Record Investigation

This section investigates the grain and variation of repeated `item_id` observations in `browse_search`. Repeated IDs are not removed: the purpose is to understand their origin and business-field behavior before making downstream modeling decisions.


In [0]:
from pyspark.sql import functions as F

# Identify item IDs that occur more than once
repeated_ids = (
    browse_df
    .groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

# Retrieve every occurrence of those items
repeated_records = browse_df.join(
    repeated_ids,
    on="item_id",
    how="inner"
)

# Compare distinct values across selected business columns
comparison_columns = [
    "title",
    "price__value",
    "price__currency",
    "seller__username",
    "condition",
    "condition_id",
]

duplicate_profile = (
    repeated_records
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        *[
            F.countDistinct(F.col(c)).alias(f"{c}_distinct")
            for c in comparison_columns
        ]
    )
)

display(
    duplicate_profile
    .orderBy(F.col("occurrences").desc())
    .limit(50)
)

In [0]:
from pyspark.sql import functions as F

(
    browse_df
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct("_dlt_load_id").alias("distinct_loads"),
        F.countDistinct("title").alias("distinct_titles"),
        F.countDistinct("price__value").alias("distinct_prices")
    )
    .filter(F.col("occurrences") > 1)
    .groupBy("distinct_loads")
    .count()
    .orderBy("distinct_loads")
    .show()
)

### 7.1 Duplicate Record and Business-Field Variation Analysis

### Objective

Investigate repeated `item_id` values in the `browse_search` dataset to determine whether they represent identical business records or multiple distinct observations of the same listing.

A repeated item identifier does not necessarily indicate a data quality issue. The same eBay listing may appear in multiple search results, and its returned attributes may differ between observations.

### Methodology

The analysis is performed at the `item_id` grain:

1. Identify item IDs occurring more than once.
2. Count distinct DLTHub load IDs for each repeated item to investigate whether repetitions occur within or across ingestion loads.
3. Exclude technical metadata (`_dlt_id`, `_dlt_load_id`) and the grouping key (`item_id`) when comparing business records.
4. Count distinct business records for each repeated item ID.

This analysis is descriptive. It does not remove records or modify the raw data.

### Observations

The profiling results returned 9,772 repeated item ID groups.

| Distinct business records per item | Number of item ID groups |
| ---------------------------------: | -----------------------: |
|                                  1 |                    5,111 |
|                                  2 |                    4,368 |
|                                  3 |                      268 |
|                                  4 |                       24 |
|                                  5 |                        1 |

All 9,772 groups in the load-level analysis had one distinct `_dlt_load_id`, indicating that the repeated observations occurred within a single ingestion load.

Of the repeated item ID groups, 5,111 had identical values across the compared business columns, while 4,661 had more than one distinct business record.

### Interpretation

The results establish that repeated item IDs cannot automatically be treated as exact duplicates.

The observations are consistent with overlapping search-query results, but the precise cause of the repetitions has not yet been established.

Differences between business records could involve price, condition, seller attributes, listing metadata, or other returned fields. The specific fields responsible for the variation remain to be investigated.

### Limitations and Next Steps

* The analysis covers the currently available raw dataset and does not establish permanent source-system behavior.
* A distinct business record does not necessarily represent a price change or a chronological listing update.
* The analysis has not yet established whether differences originate from search-query overlap, API response context, or listing changes.

**Next:** Profile field-level variation across repeated item IDs to identify which business attributes differ. Inspect representative records before defining the Silver-layer deduplication and business-grain strategy.

The raw Bronze input will remain unchanged. Any future deduplication or record-selection logic will be implemented in the appropriate downstream transformation layer.


In [0]:
from pyspark.sql import functions as F

business_columns = [
    c for c in browse_df.columns
    if c not in ["_dlt_id", "_dlt_load_id"]
]

record_variation = (
    browse_df
    .groupBy("item_id")
    .agg(
        F.count("*").alias("occurrences"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in business_columns])
        ).alias("distinct_business_records")
    )
    .filter(F.col("occurrences") > 1)
)

record_variation.groupBy("distinct_business_records") \
    .count() \
    .orderBy("distinct_business_records") \
    .show()

In [0]:
from pyspark.sql import functions as F

business_columns = [
    c for c in browse_df.columns
    if c not in ["_dlt_id", "_dlt_load_id", "item_id"]
]

repeated_ids = (
    browse_df.groupBy("item_id")
    .count()
    .filter(F.col("count") > 1)
    .select("item_id")
)

repeated_records = browse_df.join(
    repeated_ids,
    on="item_id",
    how="inner"
)

variation_expressions = [
    F.countDistinct(F.col(c)).alias(c)
    for c in business_columns
]

variation_profile = repeated_records.groupBy("item_id").agg(
    F.count("*").alias("occurrences"),
    *variation_expressions
)

varying_fields = [
    F.sum(
        F.when(F.col(c) > 1, 1).otherwise(0)
    ).alias(c)
    for c in business_columns
]

field_variation_summary = variation_profile.agg(
    *varying_fields
)

display(field_variation_summary)

### 7.2 Field-Level Variation Across Repeated Item IDs

#### Objective

Identify the business attributes responsible for distinct records among repeated `item_id` groups in `browse_search`.

The previous analysis established that repeated item IDs can contain multiple distinct business records. This analysis determines which individual fields contribute to those differences.

#### Results

The field-level variation profile identified the following business columns with non-zero variation:

| Business column               | Repeated item ID groups with variation |
| ----------------------------- | -------------------------------------: |
| `item_web_url`                |                                  4,625 |
| `priority_listing`            |                                    294 |
| `seller__feedback_score`      |                                     34 |
| `top_rated_buying_experience` |                                     14 |
| `price__value`                |                                      8 |
| `seller__feedback_percentage` |                                      1 |

All other business columns included in the profile had zero variation.

The counts represent the number of repeated item ID groups containing more than one distinct value for the respective field.

#### Interpretation

Variation among repeated records is concentrated in a limited number of business attributes.

The `item_web_url` field accounts for the largest number of varying groups. Seller feedback attributes and listing flags also differ for some repeated item IDs.

Only eight repeated item ID groups exhibit variation in `price__value`. Therefore, repeated discovery records should not automatically be interpreted as price changes.

The observed differences establish that some repeated records are not exact business-record duplicates. However, the analysis does not establish whether the differences originate from overlapping search queries, API response context, or changes to listing attributes.

#### Limitations

* The profile does not establish the chronological order of observations.
* It does not identify the specific values responsible for variation.
* It does not establish whether a difference is meaningful for downstream business analytics.
* The findings describe the current raw dataset and should not be treated as permanent source-system guarantees.

#### Next Step

Inspect representative repeated item IDs and compare their actual field values, particularly `item_web_url`, `priority_listing`, seller feedback attributes, and `price__value`.

This investigation will inform the downstream Silver-layer record-grain and deduplication strategy. No records are removed or modified during raw data profiling.



In [0]:
from pyspark.sql import functions as F

varying_item_ids = (
    repeated_records
    .groupBy("item_id")
    .agg(
        F.countDistinct("item_web_url").alias("url_variants"),
        F.countDistinct("price__value").alias("price_variants"),
        F.countDistinct("priority_listing").alias("priority_variants")
    )
    .filter(
        (F.col("url_variants") > 1) |
        (F.col("price_variants") > 1) |
        (F.col("priority_variants") > 1)
    )
    .select("item_id")
)

display(
    repeated_records
    .join(varying_item_ids, "item_id")
    .select(
        "item_id",
        "title",
        "item_web_url",
        "price__value",
        "price__currency",
        "priority_listing",
        "seller__feedback_score",
        "seller__feedback_percentage",
        "top_rated_buying_experience",
        "_dlt_load_id"
    )
    .orderBy("item_id")
    .limit(100)
)

### 7.3 Inspection of Representative Repeated Records

#### Objective

Inspect actual business-field values for repeated item IDs to understand the differences identified during field-level variation profiling.

#### Observations

Representative records show the same eBay listing appearing under different search queries.

For example, the same Xbox listing appears in searches associated with SSDs, Xbox consoles, and gaming consoles. The listing retains the same item ID, title, and price, while the `item_web_url` contains different search-related query parameters.

Similar patterns were observed for keyboard, tablet, and other product listings.

#### Interpretation

The inspected records provide evidence that overlapping search queries contribute to repeated Browse observations.

The `item_web_url` field contains search-context parameters, including `_skw`, which can differ even when the underlying listing is unchanged.

This indicates that `item_id` represents listing identity, whereas each Browse result represents a discovery observation.

The two concepts should not be treated as equivalent when designing downstream data models.


### 7.4 Investigating Price Variations Across Repeated Listings

Objective:
Investigate repeated `item_id` records where `price__value`
differs across observations.

We want to determine whether the variations represent:
- Actual listing price changes
- Differences in observation context
- Data quality issues

This investigation is exploratory. No deduplication or
business transformation is performed at this stage.

In [0]:
from pyspark.sql import functions as F

# Identify listings with more than one observed price
price_variations_df = (
    browse_df
    .groupBy("item_id")
    .agg(
        F.countDistinct("price__value").alias("distinct_prices"),
        F.collect_set("price__value").alias("observed_prices"),
        F.count("*").alias("observation_count")
    )
    .filter(F.col("distinct_prices") > 1)
    .orderBy(F.desc("distinct_prices"))
)

display(price_variations_df)

In [0]:
# Inspect all original observations for the 8 affected listings

affected_item_ids = [
    "v1|366677068914|0",
    "v1|398406025098|0",
    "v1|398406009265|0",
    "v1|168698390612|0",
    "v1|800681525774|0",
    "v1|800681290529|0",
    "v1|800681537177|0",
    "v1|147582341570|0",
]

price_investigation_df = (
    browse_df
    .filter(F.col("item_id").isin(affected_item_ids))
    .select(
        "item_id",
        "price__value",
        "price__currency",
        "item_web_url",
        "priority_listing",
        "seller__username",
        "seller__feedback_score",
        "seller__feedback_percentage",
        "_dlt_load_id"
    )
    .orderBy("item_id", "price__value")
)

display(price_investigation_df)

**Price variation investigation**

Eight listing IDs have two distinct observed prices each.

For the examined observations:

- The currency is USD.
- Seller information remains unchanged within each pair.
- Both observations share the same `_dlt_load_id`.
- Search context differs for at least some observations.

The current data does not establish whether price differences represent actual listing price changes, API response behavior, or other causes.

#### Limitations

The current exploration is based on the landed Browse Search observations. It does not establish the chronological order of repeated observations or the cause of individual attribute variations.

#### Conclusion

`item_id` identifies a listing, but repeated Browse Search rows represent multiple observations of that listing. Repeated rows should not automatically be interpreted as independent products or as chronological updates.

Further processing decisions will be made after completing the remaining exploration and assessing downstream analytical requirements.

## 8. Item Details — Completeness and Null Profiling

### Objective

Assess the completeness of the Item Details dataset and identify fields with missing values.

The objective is to understand the available data before defining Bronze and Silver processing requirements.

This stage is exploratory. No records will be removed or modified.

In [0]:
from pyspark.sql import functions as F

# Total number of records
total_rows = item_details_df.count()

# Calculate null counts and percentages for every column
null_profile_df = item_details_df.agg(
    *[
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in item_details_df.columns
    ]
)

# Convert the wide aggregation into a readable profiling table
null_profile_df = null_profile_df.select(
    F.explode(
        F.array(
            *[
                F.struct(
                    F.lit(column).alias("column_name"),
                    F.col(column).alias("null_count")
                )
                for column in item_details_df.columns
            ]
        )
    ).alias("profile")
).select("profile.*")

null_profile_df = (
    null_profile_df
    .withColumn(
        "total_rows",
        F.lit(total_rows)
    )
    .withColumn(
        "null_percentage",
        F.round(
            F.col("null_count") / F.col("total_rows") * 100,
            2
        )
    )
    .orderBy(F.desc("null_percentage"))
)

display(null_profile_df)

## 9. Item Details — Categorical Value Profiling

### Objective

Understand the distribution of categorical fields and identify distinct values, unexpected categories, and potential standardization requirements.

This is exploratory profiling. No values are modified or standardized.

In [0]:
from pyspark.sql import functions as F

categorical_columns = [
    "condition",
    "condition_id",
    "price__currency",
    "listing_marketplace_id",
    "priority_listing",
    "top_rated_buying_experience",
    "adult_only",
    "enabled_for_guest_checkout",
    "eligible_for_inline_checkout",
    "category_id",
]

for column in categorical_columns:
    print(f"Distinct values for: {column}")

    display(
        item_details_df
        .groupBy(column)
        .count()
        .orderBy(F.desc("count"))
    )

### 9.1 Findings: Categorical Value Profiling

#### Observations

| Attribute | Observed distribution |
|---|---|
| `category_id` | 2,777 distinct values |
| `listing_marketplace_id` | 8 distinct values, predominantly US |
| `condition_id` | 5 distinct values |
| `condition` | 15 distinct values |
| `price__currency` | 1 distinct value |
| Boolean attributes | Generally 2 values: true and false |
| `adult_only` | All observed values are false |

#### Condition field observations

The `condition` field contains 15 distinct textual values, although multiple values represent variations of the New condition.

The source provides different descriptions for conditions that may be semantically equivalent. Standardization may be required in a downstream business transformation, but no mapping is defined at this stage.

#### Limitations

These counts represent the dataset at the time of exploration. Item Details enrichment runs daily, so the number of records, distinct values, marketplace distribution, and null percentages are expected to change.

The observed distribution is not a fixed source contract or permanent quality baseline.

#### Conclusion

Categorical values should be preserved in Bronze as received. Any normalization of condition descriptions, marketplace selection, or category groupings will be considered during Silver and Gold modeling based on analytical requirements.

## 10. Browse Search vs Item Details Coverage

### Objective

Measure the overlap between Browse Search observations and Item Details enrichment records.

The analysis will identify:

- Listings present in both datasets.
- Listings present in Browse Search but not in Item Details.
- Listings present in Item Details but not in Browse Search.

The comparison is based on distinct listing identifiers and represents the current snapshot of the datasets.

In [0]:
# Compare listing coverage between Browse Search and Item Details

browse_ids = (
    browse_df
    .select("item_id")
    .distinct()
)

details_ids = (
    item_details_df
    .select("item_id")
    .distinct()
)

browse_count = browse_ids.count()
details_count = details_ids.count()

matched_count = (
    browse_ids
    .join(details_ids, on="item_id", how="inner")
    .count()
)

browse_only_count = (
    browse_ids
    .join(details_ids, on="item_id", how="left_anti")
    .count()
)

details_only_count = (
    details_ids
    .join(browse_ids, on="item_id", how="left_anti")
    .count()
)

print(f"Distinct Browse Search listings: {browse_count}")
print(f"Distinct Item Details listings: {details_count}")
print(f"Listings present in both: {matched_count}")
print(f"Browse Search only: {browse_only_count}")
print(f"Item Details only: {details_only_count}")

print(
    f"Browse Search enrichment coverage: "
    f"{matched_count / browse_count * 100:.2f}%"
)

## 11. Temporal Profiling

### Objective

Understand the timestamp fields available in Browse Search and Item Details and inspect their temporal ranges.

The analysis focuses on identifying:
- Available timestamp columns.
- Earliest and latest observed timestamps.
- Null values in timestamp fields.

The objective is to distinguish listing-related timestamps from ingestion or observation timestamps where the available data supports that distinction.

No timestamp transformations or business rules are applied at this stage.

In [0]:
from pyspark.sql.types import (
    TimestampType,
    DateType,
)

def get_temporal_columns(df):
    return [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            (TimestampType, DateType)
        )
    ]

browse_temporal_columns = get_temporal_columns(browse_df)
details_temporal_columns = get_temporal_columns(item_details_df)

print("Browse Search temporal columns:")
print(browse_temporal_columns)

print("\nItem Details temporal columns:")
print(details_temporal_columns)

In [0]:
# Inspect date/time-related columns and their Spark data types

for name, df in [
    ("Browse Search", browse_df),
    ("Item Details", item_details_df),
]:
    print(f"\n{name} — date/time-related columns")

    for field in df.schema.fields:
        if any(
            keyword in field.name.lower()
            for keyword in ["date", "time", "created", "updated"]
        ):
            print(f"{field.name}: {field.dataType}")

### 11.1 Findings: Temporal Field Schema

#### Observations

| Dataset | Field | Spark data type |
|---|---|---|
| Browse Search | `item_creation_date` | String |
| Browse Search | `item_end_date` | String |
| Browse Search | `item_origin_date` | String |
| Item Details | `item_creation_date` | String |
| Item Details | `item_end_date` | String |

Neither dataset contains columns currently recognized as Spark `DateType` or `TimestampType`.

#### Conclusion

The source timestamp fields are represented as strings in the current raw DataFrames.

The original values will be preserved during exploration. Timestamp parsing and validation will be considered during downstream processing, based on the source format and business requirements.

In [0]:
from pyspark.sql import functions as F

temporal_fields = {
    "Browse Search": (
        browse_df,
        [
            "item_creation_date",
            "item_end_date",
            "item_origin_date",
        ],
    ),
    "Item Details": (
        item_details_df,
        [
            "item_creation_date",
            "item_end_date",
        ],
    ),
}

for dataset_name, (df, columns) in temporal_fields.items():
    print(f"\n{dataset_name}")

    expressions = []

    for column in columns:
        parsed = F.to_timestamp(F.col(column))

        expressions.extend([
            F.min(parsed).alias(f"{column}_min"),
            F.max(parsed).alias(f"{column}_max"),
            F.sum(
                F.when(F.col(column).isNull(), 1).otherwise(0)
            ).alias(f"{column}_nulls"),
            F.sum(
                F.when(
                    F.col(column).isNotNull() & parsed.isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{column}_parse_failures"),
        ])

    display(df.agg(*expressions))

### 11.2 Findings: Temporal Range and Completeness

#### Browse Search

| Field | Minimum | Maximum | Null count |
|---|---|---|---:|
| `item_creation_date` | 2026-09-15T20:10:14Z | 2026-09-19T00:00:00Z | 0 |
| `item_end_date` | 2026-09-18T16:52:22Z | 2026-09-28T23:57:24Z | 113,502 |
| `item_origin_date` | 2012-04-14T01:04:08Z | 2026-09-19T00:00:00Z | 0 |

#### Item Details

| Field | Minimum | Maximum | Null count |
|---|---|---|---:|
| `item_creation_date` | 2026-09-18T00:00:00Z | 2026-09-19T00:00:00Z | 0 |
| `item_end_date` | 2026-09-19T04:41:41Z | 2026-09-28T23:57:24Z | 27,191 |

#### Observations

- All examined timestamp strings were successfully parsed using Spark's default timestamp parser.
- No parsing failures or null creation timestamps were observed.
- `item_end_date` is highly sparse in both datasets.
- `item_origin_date` has a substantially wider historical range than `item_creation_date`.

#### Limitations

The results represent the current data snapshot. The semantic meaning of each timestamp has not been independently verified beyond its field name and observed values.

The source timestamp strings are retained unchanged during exploration. Parsing results do not imply that the raw source representation should be overwritten.

#### Conclusion

The datasets contain usable temporal attributes for downstream analysis, but their business semantics and completeness requirements should be assessed before defining Silver-layer rules.

In [0]:
# Inspect all tables available in the current catalog and schema

display(
    spark.sql("SHOW SCHEMAS IN market_intelligence")
)